# ForestGuard

## 06 — Modelo principal de estimación de riesgo de incendio forestal

El objetivo de esta etapa es comparar diferentes algoritmos de Machine Learning para estimar la probabilidad de que se registre un incendio forestal en un cantón y periodo determinado.

La variable objetivo utilizada es `incendio`:

- `0`: no existe un incendio registrado.
- `1`: existe al menos un incendio registrado.

Los modelos utilizan información meteorológica, temporal y geográfica disponible antes de conocer el resultado.

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

## 2. Carga del dataset

Se utiliza el dataset preparado en Big Data, que integra información histórica de incendios y variables meteorológicas.

In [ ]:
from google.colab import files

archivo = "forestguard_dataset_riesgo_2022_2025.csv"

try:
    df = pd.read_csv(archivo)
except FileNotFoundError:
    print("Sube el dataset generado en Big Data:")
    uploaded = files.upload()
    nombre = list(uploaded.keys())[0]
    df = pd.read_csv(nombre)

df["fecha"] = pd.to_datetime(df["fecha"])

print("Dimensiones:", df.shape)
display(df.head())

## 3. Variable objetivo y variables predictoras

La variable objetivo es `incendio`.

Las variables predictoras fueron seleccionadas considerando que deben estar disponibles antes de conocer si se registrará un incendio.

### Variables meteorológicas
- temperatura media;
- humedad relativa media;
- precipitación;
- velocidad máxima del viento.

### Variables temporales
- mes;
- día de la semana.

### Variables geográficas
- provincia;
- cantón;
- latitud;
- longitud.

No se utilizan hectáreas afectadas, personas afectadas, animales afectados ni nivel de afectación porque estas variables corresponden a consecuencias conocidas después de ocurrido el incendio.

In [ ]:
target = "incendio"

variables_numericas = [
    "temperatura_media",
    "humedad_media",
    "precipitacion",
    "viento_max",
    "latitud",
    "longitud",
    "mes",
    "dia_semana"
]

variables_categoricas = [
    "provincia",
    "canton"
]

variables_predictoras = variables_numericas + variables_categoricas

X = df[variables_predictoras]
y = df[target]

display(y.value_counts().to_frame("Cantidad"))

## 4. División de entrenamiento y prueba

Para realizar una evaluación más cercana a un escenario real, los datos no se dividen aleatoriamente.

Se utilizan los registros correspondientes a 2022, 2023 y 2024 para entrenamiento y los registros disponibles de 2025 para evaluación.

De esta forma se analiza el comportamiento de los modelos frente a un periodo posterior al utilizado durante el entrenamiento.

In [ ]:
train_mask = df["año"].isin([2022, 2023, 2024])
test_mask = df["año"] == 2025

X_train = df.loc[train_mask, variables_predictoras]
y_train = df.loc[train_mask, target]

X_test = df.loc[test_mask, variables_predictoras]
y_test = df.loc[test_mask, target]

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\nDistribución entrenamiento:")
print(y_train.value_counts())

print("\nDistribución prueba:")
print(y_test.value_counts())

if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError(
        "El conjunto de entrenamiento o prueba no contiene ambas clases. "
        "Revisa la construcción del dataset de riesgo."
    )

## 5. Preprocesamiento

La imputación, estandarización y codificación se incorporan dentro de un `Pipeline`.

De esta manera, los transformadores se ajustan únicamente con los datos de entrenamiento y posteriormente se aplican al conjunto de prueba, reduciendo el riesgo de fuga de información.

In [ ]:
pipeline_numerico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, variables_numericas),
        ("categoricas", pipeline_categorico, variables_categoricas)
    ]
)

## 6. Modelos implementados

Se comparan tres algoritmos de clasificación para resolver el mismo problema.

1. **Regresión Logística:** modelo sencillo e interpretable que permite estimar probabilidades.
2. **Árbol de Decisión:** permite representar relaciones no lineales entre las variables.
3. **Random Forest:** combina múltiples árboles y puede capturar relaciones más complejas.

Debido al posible desbalance entre días con y sin incendios registrados, se utilizan pesos de clase balanceados.

In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Árbol de Decisión": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

## 7. Entrenamiento y evaluación

Los modelos se evalúan sobre los datos correspondientes a 2025.

Debido al posible desbalance entre las clases, no se utiliza únicamente Accuracy.

Se calculan:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- Balanced Accuracy.
- ROC-AUC.

In [ ]:
resultados = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():

    pipeline = Pipeline(
        steps=[
            ("preprocesamiento", preprocesamiento),
            ("modelo", modelo)
        ]
    )

    pipeline.fit(X_train, y_train)

    predicciones = pipeline.predict(X_test)
    probabilidades = pipeline.predict_proba(X_test)[:, 1]

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, predicciones),
        "Precision": precision_score(y_test, predicciones, zero_division=0),
        "Recall": recall_score(y_test, predicciones, zero_division=0),
        "F1": f1_score(y_test, predicciones, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predicciones),
        "ROC-AUC": roc_auc_score(y_test, probabilidades)
    })

    modelos_entrenados[nombre] = pipeline

resultados_df = pd.DataFrame(resultados)

display(
    resultados_df
    .sort_values("F1", ascending=False)
    .round(4)
)

## 8. Comparación gráfica

Se comparan las métricas obtenidas por los tres algoritmos para identificar cuál presenta el comportamiento más adecuado para el problema.

In [ ]:
resultados_df.set_index("Modelo")[
    ["Precision", "Recall", "F1", "ROC-AUC"]
].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Comparación de modelos de riesgo")
plt.ylabel("Puntuación")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

## 9. Selección del modelo

Para esta etapa se utiliza F1-score como criterio principal porque permite considerar de forma conjunta Precision y Recall.

También se revisan ROC-AUC y Balanced Accuracy antes de establecer la selección final.

In [ ]:
mejor_nombre = (
    resultados_df
    .sort_values("F1", ascending=False)
    .iloc[0]["Modelo"]
)

mejor_modelo = modelos_entrenados[mejor_nombre]

print("Mejor modelo:", mejor_nombre)

## 10. Matriz de confusión y reporte

In [ ]:
predicciones_finales = mejor_modelo.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicciones_finales,
    display_labels=[
        "Sin incendio registrado",
        "Incendio registrado"
    ]
)

plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.show()

print(
    classification_report(
        y_test,
        predicciones_finales,
        target_names=[
            "Sin incendio registrado",
            "Incendio registrado"
        ],
        zero_division=0
    )
)

## 11. Curva ROC del mejor modelo

In [ ]:
RocCurveDisplay.from_estimator(
    mejor_modelo,
    X_test,
    y_test
)

plt.title(f"Curva ROC - {mejor_nombre}")
plt.show()

## 12. Estimación de probabilidad

Además de asignar una clase, el modelo genera una probabilidad asociada a la clase `incendio = 1`.

Esta salida permite expresar el resultado como un porcentaje estimado de riesgo.

La probabilidad debe interpretarse como una estimación del modelo basada en los datos disponibles y no como una garantía de que el incendio ocurrirá.

In [ ]:
probabilidades_finales = (
    mejor_modelo
    .predict_proba(X_test)[:, 1]
)

resultados_prediccion = (
    df.loc[
        test_mask,
        ["fecha", "provincia", "canton"]
    ]
    .copy()
)

resultados_prediccion["incendio_real"] = y_test.values

resultados_prediccion["probabilidad_incendio"] = (
    probabilidades_finales * 100
)

display(
    resultados_prediccion
    .sort_values(
        "probabilidad_incendio",
        ascending=False
    )
    .head(20)
)

## 13. Clasificación visual del nivel de riesgo

Para facilitar la presentación de los resultados se transforma la probabilidad estimada en tres categorías visuales.

Estos límites se utilizan únicamente como criterio de presentación y no corresponden a clases aprendidas directamente por el modelo.

In [ ]:
def nivel_riesgo(probabilidad):
    if probabilidad <= 30:
        return "Bajo"
    elif probabilidad <= 60:
        return "Medio"
    else:
        return "Alto"

resultados_prediccion["nivel_riesgo"] = (
    resultados_prediccion[
        "probabilidad_incendio"
    ].apply(nivel_riesgo)
)

display(
    resultados_prediccion[
        [
            "fecha",
            "provincia",
            "canton",
            "probabilidad_incendio",
            "nivel_riesgo"
        ]
    ]
    .sort_values(
        "probabilidad_incendio",
        ascending=False
    )
    .head(20)
)

## 14. Conclusión

Se construyeron y compararon tres modelos de clasificación utilizando una variable objetivo relacionada directamente con el problema de ForestGuard: la existencia o ausencia de un incendio forestal registrado.

A diferencia de los análisis preliminares basados en hectáreas afectadas o nivel de afectación, este enfoque permite generar una estimación probabilística asociada a la ocurrencia de incendios.

Los modelos fueron evaluados utilizando datos de un periodo posterior al utilizado durante el entrenamiento y mediante diferentes métricas, considerando especialmente el posible desbalance de las clases.

La probabilidad obtenida debe interpretarse como una estimación basada en las variables y datos disponibles. No representa una certeza de que ocurrirá un incendio.

Las principales limitaciones siguen relacionadas con la calidad de los registros históricos, la representación geográfica utilizada y la disponibilidad de variables ambientales adicionales.